In [5]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import json
import io
import time
import os
from dotenv import load_dotenv

In [6]:
from dotenv import load_dotenv
import os

load_dotenv()

token = os.getenv('FINRA_TOKEN')
headers = {
    'Authorization': f'Bearer {token}',
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}

base_url = 'https://api.finra.org/data/group/otcMarket/name/blocksSummary'

In [7]:
# Tune these as needed
LIMIT = 5000   # max records per request (FINRA usually allows large limits)
SLEEP = 0.2    # avoid rate limiting

In [8]:

def fetch_all_finra_data():
    all_data = []
    offset = 0

    run_loop = True
    while run_loop:
        params = {
            "limit": LIMIT,
            "offset": offset
        }

        try:
            response = requests.get(base_url, params=params, headers=headers)
            response.raise_for_status()
            data = response.json()

            if response.status_code != 200:
                print(f"No more data at offset {offset}")
                run_loop = False

            all_data.extend(data)
            print(f"Fetched {len(data)} records (offset={offset})")

            offset += LIMIT
            time.sleep(SLEEP)

        except requests.exceptions.RequestException as e:
            print(f"Request failed: {e}")
            time.sleep(2)
            run_loop = False

    return pd.DataFrame(all_data)

In [9]:
df = fetch_all_finra_data()
print(f"\nTotal records: {len(df)}")

# Save to disk
df.to_pickle("finra_blocks_summary.pkl")

Fetched 3847 records (offset=0)
Fetched 3847 records (offset=5000)
Fetched 3852 records (offset=10000)
Fetched 3845 records (offset=15000)
Fetched 2404 records (offset=20000)
Request failed: 400 Client Error: Bad Request for url: https://api.finra.org/data/group/otcMarket/name/blocksSummary?limit=5000&offset=25000

Total records: 17795
